# 03 — Baseline Construction
## Classical vs Multi-Agent · Reply × LUISS 2026

**Obiettivo:** Costruire la **baseline statistica** del comportamento normale per ogni ROTTA.
La baseline viene usata nel notebook successivo per definire quanto una rotta si discosta dalla norma.

---

### Approccio

Il dataset copre ~13 mesi (dic 2023 – dic 2024) con pochissime route che hanno osservazioni mensili multiple
(max 3 mesi per rotta). Non è possibile fare un'analisi di serie temporale classica.

Usiamo invece una **baseline cross-sezionale**: per ogni feature, calcoliamo la distribuzione
su tutte le 567 rotte e definiamo soglie statistiche che separano il "normale" dall'anomalo:

| Metodo | Soglia | Quando usarlo |
|--------|--------|---------------|
| **Tukey IQR** | Q3 + 1.5 × IQR | Feature continue con distribuzione non-zero |
| **Z-score** | μ + 2.5σ | Feature con distribuzione approssimativamente normale |
| **Percentile 95°** | p95 | Feature zero-inflated (IQR = 0) |

Per ogni rotta calcoliamo:
- **Z-score** di ogni feature rispetto alla distribuzione globale
- **Flag anomalia** per feature fuori soglia
- **n_anomalie** = numero di feature anomale per rotta

---
**Output:**
- `data/processed/baseline_stats.json` — statistiche baseline per ogni feature
- `data/processed/features_with_baseline.csv` — Z-scores + flag anomalia per ogni rotta


In [1]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

ROOT     = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"

print(f"ROOT: {ROOT}")
print(f"PROC_DIR: {PROC_DIR}")


ROOT: /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent
PROC_DIR: /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent/data/processed


## 1. Caricamento dati

In [2]:
features = pd.read_csv(PROC_DIR / "features_classical.csv")

with open(PROC_DIR / "feature_cols.json") as f:
    meta = json.load(f)

print(f"features_classical.csv: {features.shape[0]} rotte × {features.shape[1]} colonne")
print(f"Feature numeriche totali per ML: {meta['n_features']}")


features_classical.csv: 567 rotte × 63 colonne
Feature numeriche totali per ML: 54


## 2. Selezione feature per la baseline

Non tutte le 54 feature numeriche sono ugualmente utili per l'anomaly detection.
Scegliamo le **11 feature più significative** dal punto di vista della sicurezza:

| Feature | Fonte | Interpretazione |
|---------|-------|----------------|
| `tot_allarmi_log` | ALLARMI | Volume di allarmi totali (log-trasformato) |
| `pct_interpol` | ALLARMI | % allarmi di origine INTERPOL |
| `pct_sdi` | ALLARMI | % allarmi di origine SDI |
| `pct_nsis` | ALLARMI | % allarmi di origine NSIS |
| `tasso_chiusura` | ALLARMI | % allarmi chiusi (gestiti) |
| `tasso_rilevanza` | ALLARMI | % allarmi rilevanti su voli con allarmi |
| `tasso_allarme_medio` | VIAGGIATORI | Tasso medio di allarmati su entrati |
| `tasso_inv_medio` | VIAGGIATORI | Tasso medio di investigati su entrati |
| `score_rischio_esiti` | VIAGGIATORI | Score rischio da esiti (respinti, fermati) |
| `tasso_respinti` | VIAGGIATORI | Frequenza di respingimenti |
| `tasso_fermati` | VIAGGIATORI | Frequenza di fermi |


In [3]:
ANOMALY_FEATURES = [
    "tot_allarmi_log",       # Volume allarmi (log-scaled, no outlier)
    "pct_interpol",          # % allarmi INTERPOL
    "pct_sdi",               # % allarmi SDI
    "pct_nsis",              # % allarmi NSIS
    "tasso_chiusura",        # % allarmi chiusi
    "tasso_rilevanza",       # % allarmi rilevanti
    "tasso_allarme_medio",   # Tasso allarme medio (VIAGGIATORI)
    "tasso_inv_medio",       # Tasso investigati medio
    "score_rischio_esiti",   # Score rischio esiti
    "tasso_respinti",        # Frequenza respinti
    "tasso_fermati",         # Frequenza fermati
]

X = features[ANOMALY_FEATURES].copy()

print(f"Feature selezionate per baseline: {len(ANOMALY_FEATURES)}")
print("\nStatistiche descrittive:")
print(X.describe().T[["mean","std","min","25%","50%","75%","max"]].to_string())


Feature selezionate per baseline: 11

Statistiche descrittive:
                      mean    std    min    25%    50%    75%     max
tot_allarmi_log     3.1694 3.0002 0.0000 0.0000 2.7081 5.9415 11.5450
pct_interpol        0.1150 0.1984 0.0000 0.0000 0.0000 0.1805  1.0000
pct_sdi             0.1293 0.2137 0.0000 0.0000 0.0000 0.2000  1.0000
pct_nsis            0.1206 0.1943 0.0000 0.0000 0.0000 0.2000  1.0000
tasso_chiusura      0.2441 0.4290 0.0000 0.0000 0.0000 0.0000  1.0000
tasso_rilevanza     0.0685 0.2276 0.0000 0.0000 0.0000 0.0000  1.0000
tasso_allarme_medio 0.2258 0.3201 0.0000 0.0000 0.0000 0.3641  1.0000
tasso_inv_medio     0.7478 0.4030 0.0000 0.5917 1.0000 1.0000  1.0000
score_rischio_esiti 0.1172 0.1656 0.0000 0.0000 0.0000 0.2000  0.6000
tasso_respinti      0.1256 0.2524 0.0000 0.0000 0.0000 0.1471  1.0000
tasso_fermati       0.1045 0.2105 0.0000 0.0000 0.0000 0.1393  1.0000


## 3. Calcolo baseline statistica per feature

In [4]:
# Per ogni feature calcola: media, std, quartili, IQR, soglie Tukey, percentili p95/p99
baseline_stats = {}

for feat in ANOMALY_FEATURES:
    col = X[feat]
    q1, q3 = col.quantile(0.25), col.quantile(0.75)
    iqr = q3 - q1
    mean, std = col.mean(), col.std()
    p95 = col.quantile(0.95)
    p99 = col.quantile(0.99)

    # Soglia superiore anomalia:
    # - Se IQR > 0: usa Tukey (Q3 + 1.5*IQR), più robusto agli outlier
    # - Se IQR = 0 (feature zero-inflated): usa percentile 95°
    if iqr > 0:
        tukey_upper = float(q3 + 1.5 * iqr)
        tukey_lower = float(q1 - 1.5 * iqr)
    else:
        # Feature con molti zeri: p95 come soglia
        tukey_upper = float(p95)
        tukey_lower = -np.inf

    baseline_stats[feat] = {
        "mean"        : float(mean),
        "std"         : float(std),
        "median"      : float(col.median()),
        "q1"          : float(q1),
        "q3"          : float(q3),
        "iqr"         : float(iqr),
        "tukey_upper" : tukey_upper,
        "tukey_lower" : tukey_lower,
        "p95"         : float(p95),
        "p99"         : float(p99),
        "z_upper"     : float(mean + 2.5 * std),
        "is_sparse"   : int(iqr == 0),
    }

baseline_df = pd.DataFrame(baseline_stats).T

print("Soglie di anomalia per feature:")
print(baseline_df[["mean","std","tukey_upper","p95","z_upper","is_sparse"]].round(4).to_string())


Soglie di anomalia per feature:
                      mean    std  tukey_upper    p95  z_upper  is_sparse
tot_allarmi_log     3.1694 3.0002      14.8537 7.6028  10.6700     0.0000
pct_interpol        0.1150 0.1984       0.4512 0.5000   0.6110     0.0000
pct_sdi             0.1293 0.2137       0.5000 0.5000   0.6635     0.0000
pct_nsis            0.1206 0.1943       0.5000 0.5000   0.6063     0.0000
tasso_chiusura      0.2441 0.4290       1.0000 1.0000   1.3168     1.0000
tasso_rilevanza     0.0685 0.2276       0.6667 0.6667   0.6375     1.0000
tasso_allarme_medio 0.2258 0.3201       0.9101 1.0000   1.0261     0.0000
tasso_inv_medio     0.7478 0.4030       1.6125 1.0000   1.7552     0.0000
score_rischio_esiti 0.1172 0.1656       0.5000 0.6000   0.5312     0.0000
tasso_respinti      0.1256 0.2524       0.3678 1.0000   0.7567     0.0000
tasso_fermati       0.1045 0.2105       0.3483 0.5000   0.6307     0.0000


## 4. Calcolo Z-scores

Per ogni rotta calcoliamo quanto ogni feature si discosta dalla media globale,
in unità di deviazioni standard.

**Z-score = (valore - media) / std**

Z-score > 2.5 → valore **molto alto** rispetto alla norma (anomalia)
Z-score < -2.5 → valore **molto basso** rispetto alla norma (anomalia in senso opposto)


In [5]:
# Z-score per ogni feature (std=0 → 1 per evitare div/0)
stds_safe = X.std().replace(0, 1.0)
Z = (X - X.mean()) / stds_safe

# Rinomina le colonne Z-score con prefisso z_
Z.columns = [f"z_{c}" for c in Z.columns]

print("Z-score — prime 5 righe:")
print(Z.head().to_string())
print(f"\nRotte con z_tot_allarmi_log > 2.5: {(Z['z_tot_allarmi_log'] > 2.5).sum()}")
print(f"Rotte con z_pct_interpol    > 2.5: {(Z['z_pct_interpol'] > 2.5).sum()}")


Z-score — prime 5 righe:
   z_tot_allarmi_log  z_pct_interpol  z_pct_sdi  z_pct_nsis  z_tasso_chiusura  z_tasso_rilevanza  z_tasso_allarme_medio  z_tasso_inv_medio  z_score_rischio_esiti  z_tasso_respinti  z_tasso_fermati
0            -1.0564         -0.5794    -0.6050     -0.6205           -0.5690            -0.3008                -0.7054             0.6258                -0.7075           -0.4976          -0.4966
1            -1.0564         -0.5794    -0.6050     -0.6205           -0.5690            -0.3008                -0.7054             0.6258                -0.7075           -0.4976          -0.4966
2            -1.0564         -0.5794    -0.6050     -0.6205           -0.5690            -0.3008                 2.4184             0.6258                -0.7075           -0.4976          -0.4966
3             0.8357         -0.5794    -0.6050      0.6662           -0.5690            -0.3008                 0.5441             0.6258                 0.5002            0.2947        

## 5. Flag anomalie per feature (metodo ibrido Tukey + Z-score)

In [6]:
# Una rotta è flaggata per una feature se supera ALMENO UNA delle due soglie:
# 1. Tukey IQR: valore > Q3 + 1.5*IQR  (o p95 per feature sparse)
# 2. Z-score:   z > 2.5

flag_df = pd.DataFrame(index=features.index)

for feat in ANOMALY_FEATURES:
    stats   = baseline_stats[feat]
    z_col   = Z[f"z_{feat}"]
    flag_tukey = features[feat] > stats["tukey_upper"]
    flag_z     = z_col > 2.5
    flag_df[f"flag_{feat}"] = (flag_tukey | flag_z).astype(int)

# Conta anomalie per rotta
flag_df["n_anomalie"]   = flag_df.filter(like="flag_").sum(axis=1)
flag_df["pct_anomalie"] = (flag_df["n_anomalie"] / len(ANOMALY_FEATURES)).round(4)

print("Distribuzione n_anomalie per rotta:")
print(flag_df["n_anomalie"].value_counts().sort_index().to_string())
print(f"\nRotte con >= 1 anomalia: {(flag_df['n_anomalie'] >= 1).sum()}")
print(f"Rotte con >= 3 anomalie: {(flag_df['n_anomalie'] >= 3).sum()}")


Distribuzione n_anomalie per rotta:
n_anomalie
0    338
1    175
2     43
3     10
4      1

Rotte con >= 1 anomalia: 229
Rotte con >= 3 anomalie: 11


## 6. Dataset con baseline integrata

Combiniamo il dataset features con i Z-scores e i flag anomalia.


In [7]:
# Unisce features + Z-scores + flag anomalie
features_with_baseline = pd.concat([features, Z, flag_df], axis=1)

print(f"features_with_baseline: {features_with_baseline.shape[0]} rotte × {features_with_baseline.shape[1]} colonne")
print(f"Null: {features_with_baseline.isnull().sum().sum()}")


features_with_baseline: 567 rotte × 87 colonne
Null: 0


## 7. Top rotte per rischio anomalie

In [8]:
# Classifica per numero di feature anomale, poi per score_composito
SUMMARY_COLS = [
    "ROTTA", "PAESE_PART", "ZONA",
    "n_anomalie", "pct_anomalie", "score_composito",
    "tot_allarmi_log", "pct_interpol", "score_rischio_esiti", "tasso_inv_medio",
]
top_anomalie = (
    features_with_baseline[SUMMARY_COLS]
    .sort_values(["n_anomalie", "score_composito"], ascending=[False, False])
    .head(20)
)
print("Top 20 rotte per anomalie rilevate:")
print(top_anomalie.to_string(index=False))


Top 20 rotte per anomalie rilevate:
  ROTTA       PAESE_PART ZONA  n_anomalie  pct_anomalie  score_composito  tot_allarmi_log  pct_interpol  score_rischio_esiti  tasso_inv_medio
RUH-VCE   Arabia Saudita  4.0           4        0.3636           0.2588           1.6094        0.0000               0.6000           1.0000
ALG-MXP          Algeria  2.0           3        0.2727           0.4848           5.1874        0.2500               0.4000           1.0000
JED-VCE   Arabia Saudita  4.0           3        0.2727           0.3775           5.3613        0.5000               0.4000           1.0000
EVN-VCE          Armenia  4.0           3        0.2727           0.3538           4.7449        0.0000               0.6000           1.0000
ASB-VCE     Turkmenistan  4.0           3        0.2727           0.2100           0.0000        0.0000               0.6000           1.0000
FIH-FCO Congo (Kinshasa)  5.0           3        0.2727           0.2100           0.0000        0.0000         

## 8. Baseline report per feature

In [9]:
sep = "=" * 68
print(sep)
print("  BASELINE REPORT — distribuzione per feature di anomalia")
print(sep)

for feat in ANOMALY_FEATURES:
    stats     = baseline_stats[feat]
    n_flagged = flag_df[f"flag_{feat}"].sum()
    pct_flag  = n_flagged / len(features) * 100
    print(f"\n  {feat}")
    print(f"    Media: {stats['mean']:.4f}  |  Std: {stats['std']:.4f}  |  Mediana: {stats['median']:.4f}")
    print(f"    Q1={stats['q1']:.4f}  Q3={stats['q3']:.4f}  IQR={stats['iqr']:.4f}")
    print(f"    Soglia anomalia: > {stats['tukey_upper']:.4f}  (sparse={bool(stats['is_sparse'])})")
    print(f"    Rotte flaggate: {n_flagged} ({pct_flag:.1f}%)")

print(f"\n{sep}")


  BASELINE REPORT — distribuzione per feature di anomalia

  tot_allarmi_log
    Media: 3.1694  |  Std: 3.0002  |  Mediana: 2.7081
    Q1=0.0000  Q3=5.9415  IQR=5.9415
    Soglia anomalia: > 14.8537  (sparse=False)
    Rotte flaggate: 2 (0.4%)

  pct_interpol
    Media: 0.1150  |  Std: 0.1984  |  Mediana: 0.0000
    Q1=0.0000  Q3=0.1805  IQR=0.1805
    Soglia anomalia: > 0.4512  (sparse=False)
    Rotte flaggate: 34 (6.0%)

  pct_sdi
    Media: 0.1293  |  Std: 0.2137  |  Mediana: 0.0000
    Q1=0.0000  Q3=0.2000  IQR=0.2000
    Soglia anomalia: > 0.5000  (sparse=False)
    Rotte flaggate: 22 (3.9%)

  pct_nsis
    Media: 0.1206  |  Std: 0.1943  |  Mediana: 0.0000
    Q1=0.0000  Q3=0.2000  IQR=0.2000
    Soglia anomalia: > 0.5000  (sparse=False)
    Rotte flaggate: 17 (3.0%)

  tasso_chiusura
    Media: 0.2441  |  Std: 0.4290  |  Mediana: 0.0000
    Q1=0.0000  Q3=0.0000  IQR=0.0000
    Soglia anomalia: > 1.0000  (sparse=True)
    Rotte flaggate: 0 (0.0%)

  tasso_rilevanza
    Media: 0.0

## 9. Salvataggio

In [10]:
# Salva features_with_baseline.csv
out_features = PROC_DIR / "features_with_baseline.csv"
features_with_baseline.to_csv(out_features, index=False)

# Salva baseline_stats.json
out_stats = PROC_DIR / "baseline_stats.json"
baseline_meta = {
    "anomaly_features"  : ANOMALY_FEATURES,
    "n_features"        : len(ANOMALY_FEATURES),
    "n_routes"          : len(features),
    "z_score_threshold" : 2.5,
    "stats"             : baseline_stats,
}
with open(out_stats, "w") as f:
    json.dump(baseline_meta, f, indent=2)

print(f"✓ features_with_baseline.csv — {len(features_with_baseline)} rotte × {features_with_baseline.shape[1]} colonne")
print(f"✓ baseline_stats.json        — {len(ANOMALY_FEATURES)} feature, soglie salvate")
print(f"\nRiepilogo anomalie:")
print(f"  Feature totali monitorate: {len(ANOMALY_FEATURES)}")
print(f"  Rotte con >= 1 anomalia:   {(flag_df['n_anomalie']>=1).sum()}")
print(f"  Rotte con >= 3 anomalie:   {(flag_df['n_anomalie']>=3).sum()}")


✓ features_with_baseline.csv — 567 rotte × 87 colonne
✓ baseline_stats.json        — 11 feature, soglie salvate

Riepilogo anomalie:
  Feature totali monitorate: 11
  Rotte con >= 1 anomalia:   229
  Rotte con >= 3 anomalie:   11


---
## Riepilogo

| Operazione | Risultato |
|-----------|-----------|
| Feature selezionate per baseline | 11 |
| Metodo soglia | Tukey IQR + Z-score (ibrido) |
| Rotte con ≥ 1 anomalia flaggata | ~226 |
| Rotte con ≥ 3 anomalie flaggate | ~11 |
| Output baseline stats | `baseline_stats.json` |
| Output dataset arricchito | `features_with_baseline.csv` |

### Prossimo notebook
**`04_anomaly_detection.ipynb`** — Applicazione dei modelli ML di anomaly detection:
- **IsolationForest** (isolamento outlier su feature ad alta dimensionalità)
- **Local Outlier Factor (LOF)** (densità locale per rotte con cluster)
- **Z-score ensemble** (combinazione dei segnali statistici della baseline)
- Ensemble dei tre modelli → score di anomalia finale per ogni rotta
